In [2]:
# Run this cell first - installs any missing packages
import subprocess
import sys

packages = ['nltk', 'flask', 'scikit-learn', 'numpy', 'pandas']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 
                           package, '-q'])

print("All packages ready!")

All packages ready!


In [5]:
import os
import re
import math
import json
import time
import pickle
import difflib
import warnings
warnings.filterwarnings('ignore')

from collections import defaultdict, Counter
from datetime import datetime

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Download required NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('punkt',     quiet=True)

print("=" * 50)
print("  All libraries imported successfully!")
print("=" * 50)

  All libraries imported successfully!


[nltk_data] Error loading stopwords: <urlopen error [Errno 8] nodename
[nltk_data]     nor servname provided, or not known>
[nltk_data] Error loading wordnet: <urlopen error [Errno 8] nodename
[nltk_data]     nor servname provided, or not known>
[nltk_data] Error loading punkt: <urlopen error [Errno 8] nodename nor
[nltk_data]     servname provided, or not known>


In [6]:
CONFIG = {
    "DATA_DIR"        : "docs",          # folder with .txt files
    "INDEX_FILE"      : "index.pkl",     # saved index file
    "TOP_K_DEFAULT"   : 5,               # default number of results
    "BM25_K1"         : 1.5,             # BM25 tuning parameter
    "BM25_B"          : 0.75,            # BM25 tuning parameter
    "SNIPPET_LENGTH"  : 250,             # characters per snippet
    "MAX_SUGGESTIONS" : 5,               # autocomplete suggestions
    "SPELL_CUTOFF"    : 0.6,             # spell correction threshold
}

print("Configuration loaded:")
for key, value in CONFIG.items():
    print(f"  {key:20s} = {value}")

Configuration loaded:
  DATA_DIR             = docs
  INDEX_FILE           = index.pkl
  TOP_K_DEFAULT        = 5
  BM25_K1              = 1.5
  BM25_B               = 0.75
  SNIPPET_LENGTH       = 250
  MAX_SUGGESTIONS      = 5
  SPELL_CUTOFF         = 0.6


In [7]:
STOP_WORDS  = set(stopwords.words('english'))
STEMMER     = PorterStemmer()
LEMMATIZER  = WordNetLemmatizer()


def clean_text(text: str) -> str:
    """Remove special characters, extra spaces."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def tokenize(text: str):
    """Split text into word tokens."""
    return re.findall(r'\b[a-z0-9]+\b', text)


def remove_stopwords(tokens):
    """Remove common English stop words."""
    return [t for t in tokens if t not in STOP_WORDS]


def stem_tokens(tokens):
    """Apply Porter Stemmer."""
    return [STEMMER.stem(t) for t in tokens]


def lemmatize_tokens(tokens):
    """Apply WordNet Lemmatizer."""
    return [LEMMATIZER.lemmatize(t) for t in tokens]


def preprocess(text: str, use_stemming=True):
    """
    Full preprocessing pipeline:
    clean → tokenize → remove stopwords → stem/lemmatize
    """
    text   = clean_text(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    if use_stemming:
        tokens = stem_tokens(tokens)
    else:
        tokens = lemmatize_tokens(tokens)
    return tokens


# ---- Quick test ----
sample = "Machine Learning algorithms learn from data automatically!"
print("Original :", sample)
print("Processed:", preprocess(sample))

Original : Machine Learning algorithms learn from data automatically!
Processed: ['machin', 'learn', 'algorithm', 'learn', 'data', 'automat']


In [8]:
def load_documents(data_dir=CONFIG["DATA_DIR"]):
    """
    Load all .txt files from data_dir.
    Returns:
        docs     : {doc_id: full_text}
        metadata : {doc_id: {title, filename, category, word_count, ...}}
    """
    docs     = {}
    metadata = {}
    doc_id   = 0

    if not os.path.exists(data_dir):
        print(f"ERROR: Folder '{data_dir}' not found!")
        print(f"Make sure your docs folder is at: {os.path.abspath(data_dir)}")
        return docs, metadata

    for filename in sorted(os.listdir(data_dir)):
        if not filename.endswith(".txt"):
            continue

        path = os.path.join(data_dir, filename)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read().strip()

        if not text:
            continue

        # Auto-detect category from filename
        name_parts = os.path.splitext(filename)[0].replace("_", " ").split()
        category   = name_parts[0].capitalize() if name_parts else "General"

        docs[doc_id] = text
        metadata[doc_id] = {
            "title"      : os.path.splitext(filename)[0].replace("_", " ").title(),
            "filename"   : filename,
            "path"       : os.path.abspath(path),
            "category"   : category,
            "word_count" : len(text.split()),
            "char_count" : len(text),
            "loaded_at"  : datetime.now().strftime("%Y-%m-%d %H:%M")
        }
        doc_id += 1

    return docs, metadata


# ---- Load documents ----
docs, doc_metadata = load_documents()

print("=" * 55)
print(f"  Documents loaded: {len(docs)}")
print("=" * 55)
print(f"\n{'ID':>4}  {'Title':<35} {'Words':>6}  Category")
print("-" * 55)
for did, meta in doc_metadata.items():
    print(f"{did:>4}  {meta['title']:<35} "
          f"{meta['word_count']:>6}  {meta['category']}")

  Documents loaded: 7

  ID  Title                                Words  Category
-------------------------------------------------------
   0  Artificial Intelligence                 56  Artificial
   1  Computer Networks                       55  Computer
   2  Data Science                            59  Data
   3  Database Systems                        52  Database
   4  Machine Learning                        60  Machine
   5  Python Programming                      54  Python
   6  Web Development                         57  Web


In [9]:
import os
print(os.getcwd())

/Users/aditrisingh


In [10]:
import os

docs_path = os.path.join(os.getcwd(), "docs")

if os.path.exists(docs_path):
    files = os.listdir(docs_path)
    print(f"docs folder found at: {docs_path}")
    print(f"Files inside ({len(files)} total):")
    for f in sorted(files):
        print(f"  {f}")
else:
    print("docs folder still not found!")
    print(f"Expected location: {docs_path}")

docs folder found at: /Users/aditrisingh/docs
Files inside (7 total):
  artificial_intelligence.txt
  computer_networks.txt
  data_science.txt
  database_systems.txt
  machine_learning.txt
  python_programming.txt
  web_development.txt


In [11]:
# ============================================================
#   MODULE 3: INVERTED INDEX
# ============================================================

def build_index(docs):
    """
    Build an inverted index from documents.

    Structure:
      index[term] = {doc_id: term_frequency}
    """
    index        = defaultdict(dict)
    doc_lengths  = {}
    term_positions = defaultdict(lambda: defaultdict(list))

    start = time.time()

    for doc_id, text in docs.items():
        tokens = preprocess(text)
        counts = Counter(tokens)
        doc_lengths[doc_id] = sum(counts.values())

        # Store term frequency
        for term, tf in counts.items():
            index[term][doc_id] = tf

        # Store term positions (for phrase search)
        for pos, token in enumerate(tokens):
            term_positions[token][doc_id].append(pos)

    elapsed = time.time() - start

    return dict(index), doc_lengths, dict(term_positions)


# ---- Build index ----
index, doc_lengths, term_positions = build_index(docs)
ALL_DOCS = set(docs.keys())

print("=" * 50)
print("  Index Built Successfully!")
print("=" * 50)
print(f"  Total documents  : {len(docs)}")
print(f"  Vocabulary size  : {len(index):,} unique terms")
print(f"  Avg doc length   : "
      f"{sum(doc_lengths.values())/len(doc_lengths):.1f} tokens")
print("=" * 50)


# Show a sample of the index
print("\nSample index entries (term → postings):")
sample_terms = list(index.keys())[:5]
for term in sample_terms:
    print(f"  '{term}' → {index[term]}")

  Index Built Successfully!
  Total documents  : 7
  Vocabulary size  : 192 unique terms
  Avg doc length   : 44.6 tokens

Sample index entries (term → postings):
  'artifici' → {0: 1, 4: 1}
  'intellig' → {0: 2, 4: 1}
  'ai' → {0: 5}
  'simul' → {0: 1}
  'human' → {0: 1}


In [12]:
# ============================================================
#   MODULE 4: BM25 RANKING ENGINE
# ============================================================

class BM25Ranker:
    """
    Best Match 25 (BM25) - an advanced TF-IDF based ranking algorithm.
    Used by real search engines like Elasticsearch.
    """

    def __init__(self, index, doc_lengths,
                 k1=CONFIG["BM25_K1"], b=CONFIG["BM25_B"]):
        self.index   = index
        self.doc_lengths = doc_lengths
        self.N       = len(doc_lengths)
        self.avgdl   = (sum(doc_lengths.values()) / float(self.N)
                        if self.N > 0 else 1.0)
        self.k1      = k1
        self.b       = b
        self.idf     = self._compute_idf()

    def _compute_idf(self):
        """Compute Inverse Document Frequency for each term."""
        idf = {}
        for term, postings in self.index.items():
            df = len(postings)
            idf[term] = math.log(
                1 + (self.N - df + 0.5) / (df + 0.5)
            )
        return idf

    def score(self, query_terms):
        """
        Score all documents against query terms using BM25.
        Returns: {doc_id: score}
        """
        scores = {}
        for term in query_terms:
            if term not in self.index:
                continue
            idf = self.idf.get(term, 0.0)
            for doc_id, tf in self.index[term].items():
                dl    = self.doc_lengths[doc_id]
                denom = (tf + self.k1 *
                         (1.0 - self.b + self.b * dl / self.avgdl))
                s = idf * tf * (self.k1 + 1.0) / denom
                scores[doc_id] = scores.get(doc_id, 0.0) + s
        return scores

    def explain_score(self, query_terms, doc_id):
        """Show how each term contributed to the score."""
        explanation = {}
        for term in query_terms:
            if term not in self.index:
                explanation[term] = 0.0
                continue
            tf = self.index[term].get(doc_id, 0)
            if tf == 0:
                explanation[term] = 0.0
                continue
            idf   = self.idf.get(term, 0.0)
            dl    = self.doc_lengths[doc_id]
            denom = (tf + self.k1 *
                     (1.0 - self.b + self.b * dl / self.avgdl))
            explanation[term] = idf * tf * (self.k1 + 1.0) / denom
        return explanation


# ---- Create ranker ----
ranker = BM25Ranker(index, doc_lengths)
print("BM25 Ranker created!")
print(f"  k1 = {ranker.k1}  (controls term frequency saturation)")
print(f"  b  = {ranker.b}   (controls document length normalization)")
print(f"  Average document length = {ranker.avgdl:.1f} tokens")

BM25 Ranker created!
  k1 = 1.5  (controls term frequency saturation)
  b  = 0.75   (controls document length normalization)
  Average document length = 44.6 tokens


In [13]:
# ============================================================
#   MODULE 5: BOOLEAN QUERY PROCESSOR
# ============================================================

def get_postings(term):
    """Get set of doc_ids containing a term (after preprocessing)."""
    processed = preprocess(term)
    if not processed:
        return set()
    stem = processed[0]
    return set(index.get(stem, {}).keys())


def boolean_search(query):
    """
    Process Boolean queries with AND, OR, NOT operators.

    Examples:
      "python AND machine"
      "database OR sql"
      "python NOT java"
      "ai AND learning NOT image"
    """
    # Split on boolean operators (keep operators)
    parts   = re.split(r'\b(AND|OR|NOT)\b', query.upper())
    tokens  = [p.strip() for p in parts if p.strip()]

    if not tokens:
        return ALL_DOCS.copy()

    result      = None
    current_op  = 'OR'          # default operator

    for token in tokens:
        if token in ('AND', 'OR', 'NOT'):
            current_op = token
            continue

        # Get original-case version of the word
        original_token = query[query.upper().find(token):
                                query.upper().find(token) + len(token)]
        docs_for_term = get_postings(original_token)

        if result is None:
            result = docs_for_term
        elif current_op == 'AND':
            result = result & docs_for_term
        elif current_op == 'OR':
            result = result | docs_for_term
        elif current_op == 'NOT':
            result = result - docs_for_term

    return result if result is not None else set()


# ---- Test Boolean search ----
print("Boolean Search Tests:")
print("-" * 40)
queries = [
    "python AND machine",
    "database OR sql",
    "python NOT database",
]
for q in queries:
    result = boolean_search(q)
    titles = [doc_metadata[d]["title"] for d in result]
    print(f"Query: '{q}'")
    print(f"  → {titles}\n")

Boolean Search Tests:
----------------------------------------
Query: 'python AND machine'
  → ['Data Science', 'Machine Learning']

Query: 'database OR sql'
  → ['Data Science', 'Database Systems', 'Web Development']

Query: 'python NOT database'
  → ['Data Science', 'Machine Learning', 'Python Programming']



In [14]:
# ============================================================
#   MODULE 6: AUTOCOMPLETE & SPELL CORRECTION
# ============================================================

# Build sorted list of all vocabulary terms
all_terms        = sorted(index.keys())
all_terms_set    = set(all_terms)

# Also keep original (unstemmed) words for better suggestions
all_raw_words = set()
for doc_text in docs.values():
    for w in re.findall(r'\b[a-z]+\b', doc_text.lower()):
        if w not in STOP_WORDS and len(w) > 2:
            all_raw_words.add(w)
all_raw_words = sorted(all_raw_words)


def autocomplete(prefix, max_suggestions=CONFIG["MAX_SUGGESTIONS"]):
    """Return words from vocabulary starting with prefix."""
    prefix  = prefix.lower().strip()
    if not prefix:
        return []
    # Match raw words (more readable than stems)
    matches = [w for w in all_raw_words if w.startswith(prefix)]
    return matches[:max_suggestions]


def spell_correct(word, cutoff=CONFIG["SPELL_CUTOFF"]):
    """
    Suggest spelling corrections using:
    1. difflib close matches against vocabulary
    2. Edit-distance heuristics
    """
    word = word.lower().strip()
    stem = STEMMER.stem(word)

    # Already correct
    if stem in all_terms_set:
        return word

    # Try raw words first (more readable)
    raw_matches = difflib.get_close_matches(
        word, all_raw_words, n=3, cutoff=cutoff
    )
    if raw_matches:
        return raw_matches[0]

    # Try stems
    stem_matches = difflib.get_close_matches(
        stem, all_terms, n=3, cutoff=cutoff
    )
    if stem_matches:
        return stem_matches[0]

    return word  # return original if no suggestion


def correct_query(query):
    """
    Correct each word in the query.
    Returns (corrected_query, was_corrected, corrections_map)
    """
    boolean_ops = {'and', 'or', 'not'}
    words       = query.lower().split()
    corrected   = []
    corrections = {}

    for word in words:
        if word in boolean_ops:
            corrected.append(word.upper())
            continue
        suggestion = spell_correct(word)
        if suggestion != word:
            corrections[word] = suggestion
        corrected.append(suggestion)

    corrected_query = " ".join(corrected)
    was_corrected   = len(corrections) > 0
    return corrected_query, was_corrected, corrections


# ---- Tests ----
print("Autocomplete Tests:")
print("-" * 40)
for prefix in ['mach', 'dat', 'pyth', 'lear']:
    suggestions = autocomplete(prefix)
    print(f"  '{prefix}...' → {suggestions}")

print("\nSpell Correction Tests:")
print("-" * 40)
misspelled_queries = [
    "machin lerning",
    "artficial intellgence",
    "databse querry",
    "pyton programing"
]
for q in misspelled_queries:
    corrected, was_fixed, fixes = correct_query(q)
    print(f"  Original : '{q}'")
    print(f"  Corrected: '{corrected}'  (fixed: {fixes})\n")

Autocomplete Tests:
----------------------------------------
  'mach...' → ['machine', 'machines']
  'dat...' → ['data', 'database', 'databases', 'datasets']
  'pyth...' → ['python']
  'lear...' → ['learn', 'learning']

Spell Correction Tests:
----------------------------------------
  Original : 'machin lerning'
  Corrected: 'machin learning'  (fixed: {'lerning': 'learning'})

  Original : 'artficial intellgence'
  Corrected: 'artificial intelligence'  (fixed: {'artficial': 'artificial', 'intellgence': 'intelligence'})

  Original : 'databse querry'
  Corrected: 'database query'  (fixed: {'databse': 'database', 'querry': 'query'})

  Original : 'pyton programing'
  Corrected: 'python programing'  (fixed: {'pyton': 'python'})



In [15]:
# ============================================================
#   MODULE 7: SNIPPET GENERATION & HIGHLIGHTING
# ============================================================

def generate_snippet(doc_text, query,
                     max_len=CONFIG["SNIPPET_LENGTH"]):
    """
    Extract the most relevant portion of a document for the query.
    Highlights matched terms with >>> <<< markers (terminal-friendly).
    """
    # Get raw query words (not stemmed) for highlighting
    raw_terms = [
        w for w in re.findall(r'\b[a-z0-9]+\b', query.lower())
        if w not in {'and', 'or', 'not'} and w not in STOP_WORDS
    ]

    if not raw_terms:
        snippet = doc_text[:max_len]
        return snippet + ("..." if len(doc_text) > max_len else "")

    # Find the best window containing most query terms
    sentences  = re.split(r'(?<=[.!?])\s+', doc_text)
    best_sent  = doc_text[:max_len]
    best_count = 0

    for sentence in sentences:
        count = sum(
            1 for t in raw_terms if t in sentence.lower()
        )
        if count > best_count:
            best_count = best_sent
            best_sent  = sentence

    # Truncate if needed
    if len(best_sent) > max_len:
        start = 0
        text_lower = best_sent.lower()
        for t in raw_terms:
            pos = text_lower.find(t)
            if pos != -1:
                start = max(0, pos - 40)
                break
        best_sent = best_sent[start:start + max_len]

    # Highlight terms with markers
    highlighted = best_sent
    for term in raw_terms:
        pattern     = re.compile(re.escape(term), re.IGNORECASE)
        highlighted = pattern.sub(
            lambda m: f"[[ {m.group(0).upper()} ]]", highlighted
        )

    return highlighted.strip() + (
        "..." if len(doc_text) > max_len else ""
    )


def generate_html_snippet(doc_text, query,
                           max_len=CONFIG["SNIPPET_LENGTH"]):
    """
    Same as generate_snippet but with HTML <mark> tags.
    Used for Flask web interface.
    """
    raw_terms = [
        w for w in re.findall(r'\b[a-z0-9]+\b', query.lower())
        if w not in {'and', 'or', 'not'} and w not in STOP_WORDS
    ]

    sentences = re.split(r'(?<=[.!?])\s+', doc_text)
    best_sent = doc_text[:max_len]
    best_count = 0

    for sentence in sentences:
        count = sum(1 for t in raw_terms if t in sentence.lower())
        if count > best_count:
            best_count = count
            best_sent  = sentence

    if len(best_sent) > max_len:
        best_sent = best_sent[:max_len]

    highlighted = best_sent
    for term in raw_terms:
        pattern     = re.compile(re.escape(term), re.IGNORECASE)
        highlighted = pattern.sub(
            lambda m: f"<mark>{m.group(0)}</mark>", highlighted
        )

    return highlighted.strip() + (
        "..." if len(doc_text) > max_len else ""
    )


# ---- Test snippet ----
sample_doc  = docs[0]
sample_q    = "machine learning algorithms"
snippet_out = generate_snippet(sample_doc, sample_q)
print("Snippet Test:")
print("-" * 50)
print(snippet_out)

TypeError: '>' not supported between instances of 'int' and 'str'

In [16]:
# ============================================================
#   MODULE 7: SNIPPET GENERATION & HIGHLIGHTING (FIXED)
# ============================================================

def generate_snippet(doc_text, query,
                     max_len=CONFIG["SNIPPET_LENGTH"]):
    """
    Extract the most relevant portion of a document for the query.
    Highlights matched terms with [[ TERM ]] markers.
    """
    # Get raw query words (not stemmed) for highlighting
    raw_terms = [
        w for w in re.findall(r'\b[a-z0-9]+\b', query.lower())
        if w not in {'and', 'or', 'not'} and w not in STOP_WORDS
    ]

    if not raw_terms:
        snippet = doc_text[:max_len]
        return snippet + ("..." if len(doc_text) > max_len else "")

    # Find the best sentence containing most query terms
    sentences  = re.split(r'(?<=[.!?])\s+', doc_text)
    best_sent  = doc_text[:max_len]
    best_count = 0                        # ← must start as integer 0

    for sentence in sentences:
        count = sum(
            1 for t in raw_terms if t in sentence.lower()
        )
        if count > best_count:            # ← now comparing int > int
            best_count = count            # ← save the COUNT not the sentence
            best_sent  = sentence         # ← save the SENTENCE separately

    # Truncate if needed
    if len(best_sent) > max_len:
        start      = 0
        text_lower = best_sent.lower()
        for t in raw_terms:
            pos = text_lower.find(t)
            if pos != -1:
                start = max(0, pos - 40)
                break
        best_sent = best_sent[start:start + max_len]

    # Highlight terms with markers
    highlighted = best_sent
    for term in raw_terms:
        pattern     = re.compile(re.escape(term), re.IGNORECASE)
        highlighted = pattern.sub(
            lambda m: f"[[ {m.group(0).upper()} ]]", highlighted
        )

    return highlighted.strip() + (
        "..." if len(doc_text) > max_len else ""
    )


def generate_html_snippet(doc_text, query,
                           max_len=CONFIG["SNIPPET_LENGTH"]):
    """
    Same as generate_snippet but with HTML mark tags.
    Used for Flask web interface.
    """
    raw_terms = [
        w for w in re.findall(r'\b[a-z0-9]+\b', query.lower())
        if w not in {'and', 'or', 'not'} and w not in STOP_WORDS
    ]

    sentences  = re.split(r'(?<=[.!?])\s+', doc_text)
    best_sent  = doc_text[:max_len]
    best_count = 0                        # ← fixed: integer not string

    for sentence in sentences:
        count = sum(1 for t in raw_terms if t in sentence.lower())
        if count > best_count:            # ← fixed: int > int
            best_count = count            # ← fixed: save count
            best_sent  = sentence         # ← save sentence

    if len(best_sent) > max_len:
        best_sent = best_sent[:max_len]

    highlighted = best_sent
    for term in raw_terms:
        pattern     = re.compile(re.escape(term), re.IGNORECASE)
        highlighted = pattern.sub(
            lambda m: f"<mark>{m.group(0)}</mark>", highlighted
        )

    return highlighted.strip() + (
        "..." if len(doc_text) > max_len else ""
    )


print("generate_snippet fixed and ready!")

# ---- Test it ----
sample_doc  = docs[0]
sample_q    = "machine learning algorithms"
snippet_out = generate_snippet(sample_doc, sample_q)
print("\nSnippet Test:")
print("-" * 50)
print(snippet_out)

generate_snippet fixed and ready!

Snippet Test:
--------------------------------------------------
AI includes [[ MACHINE ]] [[ LEARNING ]], deep [[ LEARNING ]], natural language processing, and computer vision....


In [17]:
# ============================================================
#   MODULE 8: FACETED SEARCH
# ============================================================

def get_facets():
    """Return available facet values (categories, word count ranges)."""
    categories  = sorted(set(m["category"] for m in doc_metadata.values()))
    word_counts = [m["word_count"] for m in doc_metadata.values()]

    return {
        "categories"  : categories,
        "min_words"   : min(word_counts),
        "max_words"   : max(word_counts),
        "total_docs"  : len(doc_metadata)
    }


def apply_facets(doc_ids, category=None,
                 min_words=None, max_words=None):
    """
    Filter a set of doc_ids by metadata facets.

    Parameters:
        doc_ids   : set of candidate document IDs
        category  : filter by category string (case-insensitive)
        min_words : minimum word count
        max_words : maximum word count
    """
    filtered = []
    for doc_id in doc_ids:
        meta = doc_metadata.get(doc_id, {})

        if category:
            if meta.get("category", "").lower() != category.lower():
                continue

        if min_words is not None:
            if meta.get("word_count", 0) < min_words:
                continue

        if max_words is not None:
            if meta.get("word_count", 0) > max_words:
                continue

        filtered.append(doc_id)
    return set(filtered)


# ---- Test facets ----
print("Available Facets:")
print(json.dumps(get_facets(), indent=2))

Available Facets:
{
  "categories": [
    "Artificial",
    "Computer",
    "Data",
    "Database",
    "Machine",
    "Python",
    "Web"
  ],
  "min_words": 52,
  "max_words": 60,
  "total_docs": 7
}


In [18]:
# ============================================================
#   MODULE 9: MASTER SEARCH FUNCTION
# ============================================================

def search_engine(query,
                  top_k     = CONFIG["TOP_K_DEFAULT"],
                  use_bool  = None,
                  category  = None,
                  min_words = None,
                  max_words = None,
                  verbose   = True):
    """
    Main search function combining all modules.

    Parameters:
        query     : search query string
        top_k     : number of results to return
        use_bool  : force Boolean mode (True/False/None=auto-detect)
        category  : filter by document category
        min_words : filter by minimum word count
        max_words : filter by maximum word count
        verbose   : print detailed output

    Returns:
        list of result dicts
    """
    start_time = time.time()

    if not query.strip():
        print("Please enter a search query.")
        return []

    # ---- Step 1: Spell Correction ----
    corrected_q, was_corrected, corrections = correct_query(query)

    if verbose and was_corrected:
        print(f"Did you mean: '{corrected_q}' ?  (corrections: {corrections})")

    # ---- Step 2: Detect Boolean vs Ranked ----
    bool_ops    = {'AND', 'OR', 'NOT'}
    upper_words = corrected_q.upper().split()
    has_bool    = any(op in upper_words for op in bool_ops)

    if use_bool is None:
        use_bool = has_bool

    # ---- Step 3: Preprocess query ----
    clean_q  = re.sub(r'\b(AND|OR|NOT)\b', '', corrected_q,
                      flags=re.IGNORECASE)
    tokens   = preprocess(clean_q)

    # ---- Step 4: Score documents ----
    if use_bool:
        candidate_docs = boolean_search(corrected_q)
        scores         = ranker.score(tokens)
        scores         = {d: s for d, s in scores.items()
                          if d in candidate_docs}
        # Give all boolean matches a base score if not in ranked scores
        for d in candidate_docs:
            if d not in scores:
                scores[d] = 0.01
    else:
        scores = ranker.score(tokens)

    # ---- Step 5: Apply Facet Filters ----
    if any(f is not None for f in [category, min_words, max_words]):
        filtered = apply_facets(set(scores.keys()),
                                category=category,
                                min_words=min_words,
                                max_words=max_words)
        scores = {d: s for d, s in scores.items() if d in filtered}

    # ---- Step 6: Rank Results ----
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    ranked = ranked[:top_k]

    elapsed = (time.time() - start_time) * 1000   # ms

    # ---- Step 7: Build Result Objects ----
    results = []
    for rank, (doc_id, score) in enumerate(ranked, start=1):
        meta    = doc_metadata[doc_id]
        snippet = generate_snippet(docs[doc_id], corrected_q)
        result  = {
            "rank"      : rank,
            "doc_id"    : doc_id,
            "title"     : meta["title"],
            "category"  : meta["category"],
            "word_count": meta["word_count"],
            "score"     : round(score, 4),
            "snippet"   : snippet,
            "path"      : meta["path"],
        }
        results.append(result)

    # ---- Step 8: Display Results (verbose) ----
    if verbose:
        print("=" * 65)
        print(f"  Search Results for: '{query}'")
        if was_corrected:
            print(f"  (searched as: '{corrected_q}')")
        print(f"  Mode: {'Boolean' if use_bool else 'Ranked (BM25)'}")
        print(f"  Found: {len(results)} results  |  "
              f"Time: {elapsed:.1f} ms")
        print("=" * 65)

        if not results:
            print("  No results found. Try different keywords.")
            suggestions = autocomplete(query.split()[0][:4])
            if suggestions:
                print(f"  Suggestions: {suggestions}")
        else:
            for r in results:
                print(f"\n  [{r['rank']}] {r['title']}")
                print(f"       Category : {r['category']}")
                print(f"       Score    : {r['score']}")
                print(f"       Words    : {r['word_count']}")
                print(f"       Snippet  : {r['snippet'][:120]}...")
                print()

    return results


print("Master search function ready!")

Master search function ready!


In [19]:
# ============================================================
#   TEST THE SEARCH ENGINE
# ============================================================

print("\n" + "=" * 65)
print("   SEARCH ENGINE DEMO")
print("=" * 65)

# ---- Test 1: Simple keyword search ----
results = search_engine("machine learning algorithms")

# ---- Test 2: Boolean AND ----
results = search_engine("python AND web")

# ---- Test 3: Boolean NOT ----
results = search_engine("database NOT sql")

# ---- Test 4: Spell-corrected query ----
results = search_engine("artifcial intellgence")

# ---- Test 5: Faceted search ----
results = search_engine("data science python",
                        category="Data")

# ---- Test 6: Score explanation ----
print("\nScore Explanation for 'machine learning':")
print("-" * 45)
q_tokens = preprocess("machine learning")
for doc_id in list(docs.keys())[:3]:
    explanation = ranker.explain_score(q_tokens, doc_id)
    total = sum(explanation.values())
    title = doc_metadata[doc_id]["title"]
    if total > 0:
        print(f"  {title}: {explanation}  → total={total:.4f}")


   SEARCH ENGINE DEMO
  Search Results for: 'machine learning algorithms'
  Mode: Ranked (BM25)
  Found: 3 results  |  Time: 0.4 ms

  [1] Machine Learning
       Category : Machine
       Score    : 4.7824
       Words    : 60
       Snippet  : Popular [[ MACHINE ]] [[ LEARNING ]] [[ ALGORITHMS ]] include linear regression, decision trees, and neural networks.......


  [2] Artificial Intelligence
       Category : Artificial
       Score    : 2.389
       Words    : 56
       Snippet  : AI includes [[ MACHINE ]] [[ LEARNING ]], deep [[ LEARNING ]], natural language processing, and computer vision.......


  [3] Data Science
       Category : Data
       Score    : 1.598
       Words    : 59
       Snippet  : [[ MACHINE ]] [[ LEARNING ]] is an important component of data science.......

  Search Results for: 'python AND web'
  Mode: Boolean
  Found: 2 results  |  Time: 0.1 ms

  [1] Web Development
       Category : Web
       Score    : 2.8842
       Words    : 57
       Snippet  : 

In [20]:
# ============================================================
#   BONUS: SEARCH ENGINE ANALYTICS
# ============================================================

def show_analytics():
    print("=" * 55)
    print("   SEARCH ENGINE ANALYTICS DASHBOARD")
    print("=" * 55)

    # Document stats
    print("\n Documents:")
    print(f"  Total documents    : {len(docs)}")
    print(f"  Total vocabulary   : {len(index):,} terms")
    total_tokens = sum(doc_lengths.values())
    print(f"  Total tokens       : {total_tokens:,}")
    avg_dl = total_tokens / len(docs)
    print(f"  Avg tokens per doc : {avg_dl:.1f}")

    # Top 10 most common terms
    term_freq = {term: sum(postings.values())
                 for term, postings in index.items()}
    top_terms = sorted(term_freq.items(),
                       key=lambda x: x[1], reverse=True)[:10]

    print("\n Top 10 Most Frequent Terms:")
    print(f"  {'Term':<20} {'Frequency':>10}  {'Doc Count':>10}")
    print("  " + "-" * 44)
    for term, freq in top_terms:
        df = len(index[term])
        print(f"  {term:<20} {freq:>10}  {df:>10}")

    # Category distribution
    cat_counts = Counter(m["category"] for m in doc_metadata.values())
    print("\n Category Distribution:")
    for cat, count in cat_counts.most_common():
        bar = "█" * count
        print(f"  {cat:<20} {bar} ({count})")

    # Document length distribution
    lengths = list(doc_lengths.values())
    print("\n Document Length Stats (tokens):")
    print(f"  Min    : {min(lengths)}")
    print(f"  Max    : {max(lengths)}")
    print(f"  Mean   : {sum(lengths)/len(lengths):.1f}")
    print(f"  Median : {sorted(lengths)[len(lengths)//2]}")

    print("\n" + "=" * 55)


show_analytics()

   SEARCH ENGINE ANALYTICS DASHBOARD

 Documents:
  Total documents    : 7
  Total vocabulary   : 192 terms
  Total tokens       : 312
  Avg tokens per doc : 44.6

 Top 10 Most Frequent Terms:
  Term                  Frequency   Doc Count
  --------------------------------------------
  data                         14           6
  learn                        12           3
  python                       10           4
  includ                        7           5
  databas                       7           2
  machin                        6           3
  network                       6           2
  use                           6           5
  ai                            5           1
  commun                        5           4

 Category Distribution:
  Artificial           █ (1)
  Computer             █ (1)
  Data                 █ (1)
  Database             █ (1)
  Machine              █ (1)
  Python               █ (1)
  Web                  █ (1)

 Document Length Stats (t

In [21]:
# ============================================================
#   SAVE AND LOAD INDEX
# ============================================================

def save_index(filepath=CONFIG["INDEX_FILE"]):
    data = {
        "index"          : index,
        "doc_lengths"    : doc_lengths,
        "term_positions" : term_positions,
        "docs"           : docs,
        "doc_metadata"   : doc_metadata,
    }
    with open(filepath, "wb") as f:
        pickle.dump(data, f)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"Index saved to '{filepath}' ({size_kb:.1f} KB)")


def load_index(filepath=CONFIG["INDEX_FILE"]):
    with open(filepath, "rb") as f:
        data = pickle.load(f)
    print(f"Index loaded from '{filepath}'")
    return (data["index"], data["doc_lengths"],
            data["term_positions"], data["docs"],
            data["doc_metadata"])


# Save it
save_index()

# Example: reload it later with:
# index, doc_lengths, term_positions, docs, doc_metadata = load_index()

Index saved to 'index.pkl' (12.7 KB)


In [ ]:
# ============================================================
#   INTERACTIVE SEARCH (run this cell to start searching)
# ============================================================

def interactive_search():
    print("   INTERACTIVE SEARCH ENGINE")
    print("  Commands:")
    print("    Type any query to search")
    print("    Use AND / OR / NOT for Boolean search")
    print("    Type  :suggest <prefix>  for autocomplete")
    print("    Type  :correct <word>    for spell check")
    print("    Type  :stats             for analytics")
    print("    Type  :quit              to exit")
    print("=" * 55)

    while True:
        try:
            query = input("\n  Search > ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n  Goodbye!")
            break

        if not query:
            continue

        if query.lower() == ':quit':
            print("  Goodbye!")
            break

        elif query.lower() == ':stats':
            show_analytics()

        elif query.lower().startswith(':suggest '):
            prefix      = query[9:].strip()
            suggestions = autocomplete(prefix)
            print(f"  Suggestions for '{prefix}': {suggestions}")

        elif query.lower().startswith(':correct '):
            word       = query[9:].strip()
            correction = spell_correct(word)
            print(f"  '{word}' → '{correction}'")

        else:
            search_engine(query, top_k=5)


# Start interactive search
interactive_search()

   INTERACTIVE SEARCH ENGINE
  Commands:
    Type any query to search
    Use AND / OR / NOT for Boolean search
    Type  :suggest <prefix>  for autocomplete
    Type  :correct <word>    for spell check
    Type  :stats             for analytics
    Type  :quit              to exit



  Search >  akanksha


  Search Results for: 'akanksha'
  Mode: Ranked (BM25)
  Found: 0 results  |  Time: 2.4 ms
  No results found. Try different keywords.



  Search >  Developement


  Search Results for: 'Developement'
  Mode: Ranked (BM25)
  Found: 2 results  |  Time: 0.3 ms

  [1] Web Development
       Category : Web
       Score    : 2.0821
       Words    : 57
       Snippet  : Web development involves building websites and web applications.
Frontend development uses HTML, CSS, and JavaScript to ...


  [2] Python Programming
       Category : Python
       Score    : 1.1699
       Words    : 54
       Snippet  : Python is a high-level, interpreted programming language known for simplicity.
Python supports multiple programming para...




  Search >  Artificial Intelligence


  Search Results for: 'Artificial Intelligence'
  Mode: Ranked (BM25)
  Found: 2 results  |  Time: 0.7 ms

  [1] Artificial Intelligence
       Category : Artificial
       Score    : 2.8626
       Words    : 56
       Snippet  : [[ ARTIFICIAL ]] [[ INTELLIGENCE ]] (AI) is the simulation of human [[ INTELLIGENCE ]] by machines.......


  [2] Machine Learning
       Category : Machine
       Score    : 2.2932
       Words    : 60
       Snippet  : Machine Learning is a subset of [[ ARTIFICIAL ]] [[ INTELLIGENCE ]] that enables computers to learn from data.......

